# Encoder-router probe — OS-distill cascade labels

**Question.** Does the existing encoder-router training panel learn useful routing decisions from the OS-distill cascade relabelling of the v2-100K population?

**Dataset.** `legb_pilot/os_distill_relabel/cascade_labels.parquet` contains the cascade’s final score vector, verdict provenance, and query text for 91,093 `(dataset, query_id)` rows across 46 lanes. The cascade artifact is the only label and text source; rows with blank text are explicitly excluded because they cannot yield embeddings.

**Success criteria.** Train the existing arm panel on the text-bearing cascade rows and evaluate each saved arm on a held-out, routable lane. The deployable comparison remains the global constant; the per-lane constant remains an oracle diagnostic.

**Non-goals.** No runtime merge of v2/v3 datasets, no joint parquet, no fine-tuning BGE, and no change to arm configurations or model placement.

In [7]:
from __future__ import annotations

import os
# torch, sklearn, and lightgbm each ship their own libomp on macOS; loading two
# in one process corrupts OpenMP barriers -> SIGSEGV in the next parallel op
# (seen: kernel death inside torch.ones during EncoderRouter.fit). Must be set
# BEFORE the first import of any of them.
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")

from pathlib import Path

import numpy as np
import pandas as pd

from hybrid_search_rrf_dataset.labels import AcceptabilityLabels
from encoder_router.table import KEY, MODELS_DIR, ROUTES
from encoder_router.targets import OUT_DIR
from encoder_router.training import TrainingTable


def find_data_dir(start: Path) -> Path:
    """Find the repository data root from either a repo or notebook kernel cwd."""
    for directory in (start.resolve(), *start.resolve().parents):
        candidate = directory / "src" / "data"
        if candidate.is_dir():
            return candidate
    raise FileNotFoundError(f"could not find src/data above {start}")


DATA_DIR = find_data_dir(Path.cwd())
CASCADE_LABELS = DATA_DIR / "legb_pilot" / "os_distill_relabel" / "cascade_labels.parquet"
SEED = 0

print(f"cascade labels: {'ok' if CASCADE_LABELS.exists() else 'MISSING'}  {CASCADE_LABELS}")

cascade labels: ok  /Users/andrei/projects/hybrid-search-rrf-dataset/src/data/legb_pilot/os_distill_relabel/cascade_labels.parquet


## Plan

**Metric — decisive-headroom capture (per arm × held-out lane):**

$$H = \frac{\text{captured} - \text{const}}{\text{oracle} - \text{const}}$$

The deployable bar is the global best constant computed from the training lanes. The per-lane best constant is retained only as an oracle diagnostic: it assumes a lane label at serving time. Rows where the route scores do not differ carry no dense-vs-sparse routing signal and are excluded by the existing decisive-row evaluation.

**Probe arms — unchanged.** Every variant retains `corpus_branch=False` except `design_branches`, so the panel continues to separate the base MLP, shuffled-target floor, taxonomy inputs, Zipf inputs, LightGBM, and the existing design branch. Each arm is fit once with `FAIR_LANE` held out and saved in the established model directory.

## 1. Load cascade labels and validate query text

The cascade artifact is the sole source of scores, verdict provenance, and query text. IDs are normalized to strings for the downstream embedding cache and catalog joins. Rows with absent or blank query text are reported and excluded explicitly.

In [8]:
cascade_raw = pd.read_parquet(CASCADE_LABELS).copy()
cascade_raw["query_id"] = cascade_raw["query_id"].astype(str)

required = {"dataset", "query_id", "label_layer", *(f"score_{route}" for route in ROUTES)}
missing_columns = required.difference(cascade_raw.columns)
assert not missing_columns, f"cascade labels missing columns: {sorted(missing_columns)}"
assert not cascade_raw.duplicated(KEY).any(), "cascade labels must be unique on (dataset, query_id)"

print(f"cascade rows: {len(cascade_raw):,} across {cascade_raw['dataset'].nunique()} lanes")
display(cascade_raw["label_layer"].value_counts().rename_axis("label_layer").to_frame("rows"))

cascade rows: 91,093 across 46 lanes


,rows
label_layer,
l1,42871
judge_queue_tied,17252
argmax_l1_l2,8132
unmeasured_tied,8098
judge_queue_zero,6728
depth100_fallback,5044
unmeasured_zero,2968


In [9]:
has_text = cascade_raw["query"].notna() & cascade_raw["query"].astype(str).str.strip().ne("")
text_missing = cascade_raw.loc[~has_text].groupby("dataset").size().sort_values(ascending=False)

print(f"query text available: {int(has_text.sum()):,}/{len(cascade_raw):,} rows")
print(f"excluded (blank query text): {int((~has_text).sum()):,} rows")
display(text_missing.rename("rows_without_text").to_frame())

cascade_trainable = cascade_raw.loc[has_text].reset_index(drop=True)
assert cascade_trainable["query"].notna().all()
assert cascade_trainable["query"].astype(str).str.strip().ne("").all()

query text available: 91,080/91,093 rows
excluded (blank query text): 13 rows


,rows_without_text
dataset,
crumb-code-retrieval,13


## 2. Build the training table from the cascade labels

`AcceptabilityLabels` derives the existing `ok_*` targets, the cost-aware serving decision, and the outcome shape from the cascade score vector — nothing here depends on the compact cascade schema carrying a `shape` column.

In [10]:
def build_table(cascade_labels: pd.DataFrame) -> TrainingTable:
    """Inject valid, text-bearing cascade rows into TrainingTable without writing a new parquet."""
    merged = AcceptabilityLabels(cascade_labels, tolerance=None).frame()
    table = TrainingTable()
    table.__dict__["frame"] = merged.reset_index(drop=True)
    return table

table = build_table(cascade_trainable)
print(f"training rows: {len(table.frame):,} across {table.frame['dataset'].nunique()} lanes")
print(table.frame["shape"].value_counts().to_string())

training rows: 91,080 across 46 lanes
shape
routes_differ    57456
all_tied         23941
all_zero          9683


In [11]:
# The compact cascade artifact is intentionally score-first. Verify the reconstructed
# training frame has the inputs and targets the downstream arms require.
required_training_columns = {"query", "shape", "serve", *(f"ok_{route}" for route in ROUTES)}
missing_training_columns = required_training_columns.difference(table.frame.columns)
assert not missing_training_columns, f"training frame missing: {sorted(missing_training_columns)}"
assert not table.frame.duplicated(KEY).any()
table.frame[KEY + ["query", "label_layer", "shape", "serve"]].head()

,dataset,query_id,query,label_layer,shape,serve
0,clerc,100288,§ 2L1.2. The Government asserts that Rosales w...,judge_queue_tied,all_tied,sparse_only
1,clerc,101226,"415 U.S. 989, 94 S.Ct. 1586, 39 L.Ed.2d 885 (1...",judge_queue_tied,all_tied,sparse_only
2,clerc,101912,that exists independently of the breached cont...,argmax_l1_l2,routes_differ,sparse_only
3,clerc,102197,1631. Section 1631 provides that a district co...,judge_queue_tied,all_tied,sparse_only
4,clerc,103920,"at Exh. 1-2, 6, 8-13, 16], As a result, the Co...",argmax_l1_l2,routes_differ,sparse_only


## 3. Probe arms — 6 configurations

Six arm configurations are trained once and saved in §3b: `no_branches`, `shuffled_targets`, `features_input_nocorpus`, `zipf_input_nocorpus`, `lightgbm_nocorpus`, and `design_branches`. Their inputs, branches, learners, and held-out evaluation protocol are unchanged from the source notebook.

In [12]:
from IPython.display import display
from encoder_router.evaluate import Arm

PROBE_ARMS = (
    Arm("no_branches",             cell_branch=False, corpus_branch=False),
    Arm("shuffled_targets",        shuffle_targets=True, corpus_branch=False),
    Arm("features_input_nocorpus", feature_inputs=True, cell_branch=False, corpus_branch=False),
    Arm("zipf_input_nocorpus",     zipf_inputs=True,    cell_branch=False, corpus_branch=False),
    Arm("lightgbm_nocorpus",       feature_inputs=True, learner="lgbm",
                                   cell_branch=False, corpus_branch=False),
    # LUPI: predict corpus/gold-doc profiles from the query latent (a "hidden corpus
    # representation") and feed it forward to the route layers — serve-safe (no corpus at
    # inference). The one arm that can carry collection-relative signal query-only.
    Arm("design_branches",         cell_branch=True, corpus_branch=True),
)

def _with_derived(df: pd.DataFrame) -> pd.DataFrame:
    """Derive headroom over the per-lane BEST constant (+ served_max) for the eye tests."""
    for col in ("train_val_loss", "train_epochs"):
        if col not in df.columns:
            df = df.assign(**{col: np.nan})
    const_cols = [c for c in ("const_dense", "const_sparse", "const_rrf") if c in df.columns]
    best_const = df[const_cols].max(axis=1)
    served_cols = [c for c in ("served_dense_only", "served_sparse_only", "served_pure_rrf") if c in df.columns]
    served_max = df[served_cols].max(axis=1)
    gap = (df["oracle"] - best_const).replace(0, np.nan)
    return df.assign(
        best_const   = best_const,
        served_max   = served_max,
        headroom     = (df["captured"] - best_const) / gap,
        oracle_ratio = df["captured"] / df["oracle"].replace(0, np.nan),
    )

## 3a. Pick the validation lane FIRST — held out during training (no leakage)

Rank the cascade’s available lanes by routable headroom and route diversity, select `FAIR_LANE`, and exclude it from §3b fitting. The saved arm records the holdout in `meta.json`; §4e evaluates that exact artifact on the held-out lane.

In [13]:
# 4e-pre — rank lanes by routable headroom (oracle - best_const) and route diversity
MIN_GAP = 0.10             # routable headroom over the best constant a lane must offer
MAX_DOMINANT_SHARE = 0.55  # truth routes must be balanced: no single route may exceed this share
MIN_ROWS = 300             # enough decisive rows for a stable held-out estimate

def rank_lanes(table, min_rows=200):
    dec = table.frame[(table.frame["shape"] == "routes_differ") & table.frame["serve"].notna()]
    rows = []
    for lane, g in dec.groupby("dataset"):
        if len(g) < min_rows:
            continue
        sc = {r: g[f"score_{r}"].to_numpy() for r in ROUTES}
        consts = {r: float(sc[r].mean()) for r in ROUTES}
        best_const = max(consts.values())
        oracle = float(np.column_stack([sc[r] for r in ROUTES]).max(1).mean())
        mix = g["serve"].value_counts(normalize=True)
        rows.append({"lane": lane, "n": len(g), "best_const": best_const, "oracle": oracle,
                     "routable_gap": oracle - best_const,
                     "best_route": max(consts, key=consts.get),
                     "dominant_serve": mix.idxmax(), "dominant_share": float(mix.max())})
    return pd.DataFrame(rows).sort_values("routable_gap", ascending=False).reset_index(drop=True)

ranked = rank_lanes(table)
print("lanes by ROUTABLE headroom (big gap + low dominant_share = fair router test):")
display(ranked.round(3).head(15))
print("clerc (the bad default) for contrast:")
display(ranked[ranked["lane"] == "clerc"].round(3))

_fair = ranked[(ranked["routable_gap"] > MIN_GAP)
               & (ranked["dominant_share"] < MAX_DOMINANT_SHARE)
               & (ranked["n"] >= MIN_ROWS)]
FAIR_LANE = _fair.iloc[0]["lane"] if len(_fair) else ranked.iloc[0]["lane"]
print(f"\nfair candidates (gap>{MIN_GAP}, dominant_share<{MAX_DOMINANT_SHARE}, n>={MIN_ROWS}): "
      f"{_fair['lane'].tolist()[:8]}")
print(f"-> FAIR_LANE = {FAIR_LANE!r}  (used by §3b/§4e; override freely, or loop over _fair['lane'])")

lanes by ROUTABLE headroom (big gap + low dominant_share = fair router test):


,lane,n,best_const,oracle,routable_gap,best_route,dominant_serve,dominant_share
0,rarb-math,4156,0.492,0.725,0.234,pure_rrf,sparse_only,0.590
1,techqa,214,0.537,0.742,0.205,pure_rrf,sparse_only,0.621
2,msmarco-passage-dev,849,0.643,0.804,0.161,dense_only,dense_only,0.555
3,finder,4569,0.336,0.490,0.153,dense_only,sparse_only,0.743
4,quest,9669,0.545,0.683,0.138,sparse_only,sparse_only,0.849
5,scirgen-geo-en,11255,0.274,0.413,0.138,pure_rrf,sparse_only,0.877
6,orcas,2185,0.609,0.743,0.134,dense_only,sparse_only,0.629
7,webfaq-eng,2548,0.627,0.759,0.132,dense_only,dense_only,0.572
8,crumb-set-operation-entity-retrieval,364,0.510,0.619,0.109,pure_rrf,sparse_only,0.857
9,antique,209,0.656,0.763,0.107,pure_rrf,sparse_only,0.732


clerc (the bad default) for contrast:


,lane,n,best_const,oracle,routable_gap,best_route,dominant_serve,dominant_share
13,clerc,1539,0.631,0.712,0.081,sparse_only,sparse_only,0.908



fair candidates (gap>0.1, dominant_share<0.55, n>=300): []
-> FAIR_LANE = 'rarb-math'  (used by §3b/§4e; override freely, or loop over _fair['lane'])


## 3b. Train & save ONE model per arm — holding out FAIR_LANE

Per arm, fit ONE model on all cascade-training rows except `FAIR_LANE` and save to `BASE/<arm.name>/`. The held-out lane is recorded in `meta.json`, so §4e validates the exact saved model with no leakage. `FORCE_RETRAIN=True` overwrites; re-runs otherwise skip matching cached arms.

In [14]:
# 3b — per-arm: train ONE model per arm on all lanes EXCEPT FAIR_LANE (held out), save each
import json as _json, joblib, time as _time
from pathlib import Path as _P
from encoder_router.evaluate import LaneCV, tuned_thresholds
from encoder_router.model import EncoderRouter
from encoder_router.table import (EMBEDDING_PREFIXES, HEAD_ROUTES, NgramSvd,
                                   ZipfStats, serve_from_probabilities)
from encoder_router.training import QueryEmbeddings

TRAIN_VERSION = "2026-09-08-os-distill-cascade-v2-artifact-query"   # bump on ANY training-logic change -> invalidates cache

def _zstats(a):
    m = a.mean(0); s = a.std(0); s[s == 0] = 1.0
    return m.astype(np.float32), s.astype(np.float32)

def train_arm(table, arm, base_dir, holdout_lane=None, seed=SEED):
    """Train ONE model faithful to `arm` on all lanes EXCEPT holdout_lane; save to base_dir/arm.name."""
    t0 = _time.perf_counter()
    def log(msg): print(f"  [{arm.name}] +{_time.perf_counter()-t0:6.1f}s  {msg}", flush=True)

    frame = table.frame
    pool_mask = np.ones(len(frame), bool) if holdout_lane is None else (frame["dataset"] != holdout_lane).to_numpy()
    pool = np.flatnonzero(pool_mask)
    log(f"START learner={arm.learner} feature_inputs={arm.feature_inputs} zipf_inputs={arm.zipf_inputs}")
    log(f"holdout_lane={holdout_lane!r} | train on {len(pool):,}/{len(frame):,} rows "
        f"({frame.loc[pool_mask, 'dataset'].nunique()} lanes)")
    emb = QueryEmbeddings(arm.embedding_model).matrix(frame); log(f"embeddings {emb.shape}")
    svd = NgramSvd(seed=seed).fit(frame.loc[pool_mask, "query"]); log("ngram-SVD fit (train lanes only)")
    blocks, stats = [emb, svd.transform(frame["query"])], {}
    if arm.feature_inputs:
        log("assembling taxonomy feature inputs...")
        f = table.feature_matrix.to_numpy(np.float32); m, s = _zstats(f[pool_mask])
        stats["feat"] = (m, s); blocks.append((f - m) / s); log(f"feature inputs {f.shape}")
    if arm.zipf_inputs:
        z = ZipfStats().frame(frame["query"]).to_numpy(np.float32); m, s = _zstats(z[pool_mask])
        stats["zipf"] = (m, s); blocks.append((z - m) / s); log(f"zipf inputs {z.shape}")
    x = np.concatenate(blocks, axis=1).astype(np.float32)
    route = table.route_targets().to_numpy(np.float32)
    log(f"input matrix x={x.shape}")
    cell_t, corpus_t, feat_t = LaneCV(table, seed=seed)._targets(arm, pool_mask)
    log(f"branches: cell={None if cell_t is None else cell_t.shape} "
        f"corpus={None if corpus_t is None else corpus_t.shape} feature={None if feat_t is None else feat_t.shape}")
    if arm.shuffle_targets:                     # real permutation FLOOR: shuffle ROUTE labels in-pool
        rp = np.random.default_rng(seed + 1).permutation(len(pool))
        route = route.copy(); route[pool] = route[pool][rp]
        log("shuffled ROUTE labels within train pool (permutation floor; thresholds still use real labels)")

    p = _P(base_dir) / arm.name; p.mkdir(parents=True, exist_ok=True)
    if arm.learner == "lgbm":
        from lightgbm import LGBMClassifier
        models = []
        for i, head in enumerate(HEAD_ROUTES):
            ok = (~np.isnan(route[:, i])) & pool_mask; pos = route[ok, i].sum()
            log(f"lgbm head '{head}': fit on {int(ok.sum()):,} rows ({int(pos):,} positive)")
            models.append(LGBMClassifier(n_estimators=400, learning_rate=0.05, n_jobs=1,
                random_state=seed, verbose=-1,
                scale_pos_weight=float(np.clip((ok.sum() - pos) / max(pos, 1), 1, 100))
                ).fit(x[ok], route[ok, i]))
        joblib.dump(models, p / "lgbm.joblib"); log("lgbm models saved")
        probs_tr = pd.DataFrame(np.column_stack([m.predict_proba(x[pool])[:, 1] for m in models]), columns=HEAD_ROUTES)
    else:
        rng = np.random.default_rng(seed); perm = rng.permutation(len(pool)); cut = max(len(pool) // 10, 1)
        fi, vi = pool[perm[cut:]], pool[perm[:cut]]
        log(f"mlp fit: {len(fi):,} train / {len(vi):,} val  (fit bar tracks epochs)")
        er = EncoderRouter(seed=seed).fit(x[fi], route[fi],
            None if cell_t is None else cell_t[fi], None if corpus_t is None else corpus_t[fi],
            None if feat_t is None else feat_t[fi],
            x_val=x[vi], val_route_targets=route[vi])
        log(f"mlp done: {len(er.history)} epochs, best_val={getattr(er, 'best_val_loss', float('nan')):.4f}")
        er.save(p / "router.pt"); log("router.pt saved"); probs_tr = er.probabilities(x[pool])
    thr = tuned_thresholds(probs_tr, frame.loc[pool_mask]); log(f"tuned thresholds {np.round(thr, 3).tolist()}")
    svd.save(p / "svd.joblib"); np.save(p / "thresholds.npy", thr)
    for k, (m, s) in stats.items():
        np.save(p / f"{k}_mean.npy", m); np.save(p / f"{k}_std.npy", s)
    mix = pd.Series(serve_from_probabilities(probs_tr, thr)).value_counts(normalize=True).round(2).to_dict()
    log(f"served mix (train lanes): {mix}")
    (p / "meta.json").write_text(_json.dumps(
        {"arm": arm.name, "learner": arm.learner, "embedding_model": arm.embedding_model,
         "prefix": EMBEDDING_PREFIXES.get(arm.embedding_model, ""),
         "feature_inputs": arm.feature_inputs, "zipf_inputs": arm.zipf_inputs,
         "serve_safe": not arm.feature_inputs, "holdout_lane": holdout_lane,
         "trained_at": _time.strftime("%Y-%m-%d %H:%M:%S"), "train_version": TRAIN_VERSION}))
    log(f"SAVED -> {p}  (held out {holdout_lane!r}, total {_time.perf_counter()-t0:.1f}s)")
    return p

BASE = MODELS_DIR / "classifiers_union_200k"     # established result placement; per-arm models under BASE/<arm.name>/
FORCE_RETRAIN = False                          # True -> retrain & OVERWRITE every arm, even if saved

def _cache_ok(arm, base, holdout):
    """A saved arm is reusable ONLY if its manifest matches this run's holdout lane + config.
    A bare exists() check would silently reuse pre-holdout (all-lanes, leaked) models."""
    p = base / arm.name; mp = p / "meta.json"
    if not mp.exists():
        return False
    try:
        m = _json.loads(mp.read_text())
    except Exception:
        return False
    weights = "lgbm.joblib" if arm.learner == "lgbm" else "router.pt"
    return (m.get("train_version") == TRAIN_VERSION and m.get("holdout_lane") == holdout
            and m.get("arm") == arm.name and m.get("learner") == arm.learner
            and m.get("feature_inputs") == arm.feature_inputs
            and m.get("zipf_inputs") == arm.zipf_inputs and (p / weights).exists())

_todo   = list(PROBE_ARMS) if FORCE_RETRAIN else [a for a in PROBE_ARMS if not _cache_ok(a, BASE, FAIR_LANE)]
_cached = [] if FORCE_RETRAIN else [a.name for a in PROBE_ARMS if _cache_ok(a, BASE, FAIR_LANE)]
print(f"per-arm training | holdout={FAIR_LANE!r} | cascade table {len(table.frame):,} rows | FORCE_RETRAIN={FORCE_RETRAIN}")
print(f"  cached (skip): {_cached or '-'}")
print(f"  to train:      {[a.name for a in _todo] or '-'}", flush=True)
for _n, _a in enumerate(_todo, 1):
    print(f"===== arm {_n}/{len(_todo)}: {_a.name} =====", flush=True)
    train_arm(table, _a, BASE, holdout_lane=FAIR_LANE)
    print(flush=True)
print("all arms saved." if _todo else "nothing to train - all arms cached.")

per-arm training | holdout='rarb-math' | cascade table 91,080 rows | FORCE_RETRAIN=False
  cached (skip): -
  to train:      ['no_branches', 'shuffled_targets', 'features_input_nocorpus', 'zipf_input_nocorpus', 'lightgbm_nocorpus', 'design_branches']
===== arm 1/6: no_branches =====
  [no_branches] +   0.0s  START learner=mlp feature_inputs=False zipf_inputs=False
  [no_branches] +   0.0s  holdout_lane='rarb-math' | train on 84,404/91,080 rows (45 lanes)
  [no_branches] +   2.7s  embeddings (91080, 384)
  [no_branches] +  16.9s  ngram-SVD fit (train lanes only)
  [no_branches] +  23.1s  input matrix x=(91080, 512)
  [no_branches] +  23.1s  branches: cell=None corpus=None feature=None
  [no_branches] +  23.1s  mlp fit: 75,964 train / 8,440 val  (fit bar tracks epochs)


fit:   0%|          | 0/200 [00:00<?, ?it/s]

  [no_branches] +  29.1s  mlp done: 26 epochs, best_val=0.2067
  [no_branches] +  29.1s  router.pt saved
  [no_branches] +  29.2s  tuned thresholds [0.35, 0.9]
  [no_branches] +  29.2s  served mix (train lanes): {'sparse_only': 0.84, 'dense_only': 0.16}
  [no_branches] +  29.2s  SAVED -> /Users/andrei/projects/hybrid-search-rrf-dataset/models/classifiers_union_200k/no_branches  (held out 'rarb-math', total 29.2s)

===== arm 2/6: shuffled_targets =====
  [shuffled_targets] +   0.0s  START learner=mlp feature_inputs=False zipf_inputs=False
  [shuffled_targets] +   0.0s  holdout_lane='rarb-math' | train on 84,404/91,080 rows (45 lanes)
  [shuffled_targets] +   2.6s  embeddings (91080, 384)
  [shuffled_targets] +  16.9s  ngram-SVD fit (train lanes only)
  [shuffled_targets] +  23.1s  input matrix x=(91080, 512)
extracting features for 16676 unindexed queries
  [shuffled_targets] +  45.6s  branches: cell=(91080, 44) corpus=None feature=None
  [shuffled_targets] +  45.6s  shuffled ROUTE labe

fit:   0%|          | 0/200 [00:00<?, ?it/s]

  [shuffled_targets] +  52.8s  mlp done: 22 epochs, best_val=0.2454
  [shuffled_targets] +  52.8s  router.pt saved
  [shuffled_targets] +  53.0s  tuned thresholds [0.3, 0.55]
  [shuffled_targets] +  53.0s  served mix (train lanes): {'sparse_only': 1.0}
  [shuffled_targets] +  53.0s  SAVED -> /Users/andrei/projects/hybrid-search-rrf-dataset/models/classifiers_union_200k/shuffled_targets  (held out 'rarb-math', total 53.0s)

===== arm 3/6: features_input_nocorpus =====
  [features_input_nocorpus] +   0.0s  START learner=mlp feature_inputs=True zipf_inputs=False
  [features_input_nocorpus] +   0.0s  holdout_lane='rarb-math' | train on 84,404/91,080 rows (45 lanes)
  [features_input_nocorpus] +   2.5s  embeddings (91080, 384)
  [features_input_nocorpus] +  17.0s  ngram-SVD fit (train lanes only)
  [features_input_nocorpus] +  23.2s  assembling taxonomy feature inputs...
  [features_input_nocorpus] +  23.3s  feature inputs (91080, 83)
  [features_input_nocorpus] +  23.3s  input matrix x=(91

...

  [features_input_nocorpus] +  29.8s  mlp done: 27 epochs, best_val=0.2085
  [features_input_nocorpus] +  29.8s  router.pt saved
  [features_input_nocorpus] +  30.0s  tuned thresholds [0.35, 0.9]
  [features_input_nocorpus] +  30.0s  served mix (train lanes): {'sparse_only': 0.82, 'dense_only': 0.18}
  [features_input_nocorpus] +  30.0s  SAVED -> /Users/andrei/projects/hybrid-search-rrf-dataset/models/classifiers_union_200k/features_input_nocorpus  (held out 'rarb-math', total 30.0s)

===== arm 4/6: zipf_input_nocorpus =====
  [zipf_input_nocorpus] +   0.0s  START learner=mlp feature_inputs=False zipf_inputs=True
  [zipf_input_nocorpus] +   0.0s  holdout_lane='rarb-math' | train on 84,404/91,080 rows (45 lanes)
  [zipf_input_nocorpus] +   2.6s  embeddings (91080, 384)
  [zipf_input_nocorpus] +  16.8s  ngram-SVD fit (train lanes only)
  [zipf_input_nocorpus] +  25.1s  zipf inputs (91080, 5)
  [zipf_input_nocorpus] +  25.1s  input matrix x=(91080, 517)
  [zipf_input_nocorpus] +  25.1s  b

fit:   0%|          | 0/200 [00:00<?, ?it/s]

  [zipf_input_nocorpus] +  31.1s  mlp done: 27 epochs, best_val=0.2074
  [zipf_input_nocorpus] +  31.1s  router.pt saved
  [zipf_input_nocorpus] +  31.3s  tuned thresholds [0.4, 0.85]
  [zipf_input_nocorpus] +  31.3s  served mix (train lanes): {'sparse_only': 0.75, 'dense_only': 0.25}
  [zipf_input_nocorpus] +  31.3s  SAVED -> /Users/andrei/projects/hybrid-search-rrf-dataset/models/classifiers_union_200k/zipf_input_nocorpus  (held out 'rarb-math', total 31.3s)

===== arm 5/6: lightgbm_nocorpus =====
  [lightgbm_nocorpus] +   0.0s  START learner=lgbm feature_inputs=True zipf_inputs=False
  [lightgbm_nocorpus] +   0.0s  holdout_lane='rarb-math' | train on 84,404/91,080 rows (45 lanes)
  [lightgbm_nocorpus] +   2.8s  embeddings (91080, 384)
  [lightgbm_nocorpus] +  17.5s  ngram-SVD fit (train lanes only)
  [lightgbm_nocorpus] +  23.7s  assembling taxonomy feature inputs...
  [lightgbm_nocorpus] +  23.8s  feature inputs (91080, 83)
  [lightgbm_nocorpus] +  24.0s  input matrix x=(91080, 595

fit:   0%|          | 0/200 [00:00<?, ?it/s]

  [design_branches] +  37.0s  mlp done: 40 epochs, best_val=0.2056
  [design_branches] +  37.0s  router.pt saved
  [design_branches] +  37.1s  tuned thresholds [0.35, 0.8]
  [design_branches] +  37.1s  served mix (train lanes): {'sparse_only': 0.78, 'dense_only': 0.22}
  [design_branches] +  37.1s  SAVED -> /Users/andrei/projects/hybrid-search-rrf-dataset/models/classifiers_union_200k/design_branches  (held out 'rarb-math', total 37.1s)

all arms saved.


## 4b. Eye test — inspect the cascade lanes and real query errors

First inspect the available training population by lane and label layer. Then refit one leave-one-lane-out fold for a selected lane and read the decisive rows where the router disagrees with the score-derived serving route. There is no baseline-vs-union delta here: this notebook has one label source only.

In [15]:
# EYE TEST A — cascade coverage by lane and label layer
lane_population = (
    table.frame.groupby("dataset")
    .agg(
        rows=("query_id", "size"),
        decisive=("shape", lambda s: int((s == "routes_differ").sum())),
        labelled=("serve", lambda s: int(s.notna().sum())),
    )
    .assign(decisive_share=lambda d: d["decisive"] / d["rows"])
    .sort_values(["decisive", "rows"], ascending=False)
)
display(lane_population.head(20).round(3))
print("label-layer mix among trainable cascade rows:")
display(table.frame["label_layer"].value_counts().rename("rows").to_frame())

,rows,decisive,labelled,decisive_share
dataset,,,,
scirgen-geo-en,20000,11255,14859,0.563
quest,17377,9669,16557,0.556
crumb-legal-qa,6999,6148,6342,0.878
finder,5703,4569,4981,0.801
rarb-math,6676,4156,6623,0.623
crumb-code-retrieval,3885,3644,3654,0.938
gooaq,5676,3093,5657,0.545
webfaq-eng,6872,2548,6867,0.371
orcas,3065,2185,2979,0.713


label-layer mix among trainable cascade rows:


,rows
label_layer,
l1,42871
judge_queue_tied,17252
argmax_l1_l2,8132
unmeasured_tied,8098
judge_queue_zero,6728
depth100_fallback,5044
unmeasured_zero,2955


In [16]:
# EYE TEST B — read the real queries the router gets wrong on a chosen lane (one refit)
from encoder_router.evaluate import LaneCV
from encoder_router.training import QueryEmbeddings

def query_drilldown(table, lane, arm=PROBE_ARMS[0], seed=SEED):
    """Refit one LOO fold and return per-query rows with truth vs router serve."""
    cv = LaneCV(table, seed=seed)
    frame = table.frame
    emb = QueryEmbeddings(arm.embedding_model).matrix(frame)
    train = (frame["dataset"] != lane).to_numpy()
    x = cv._inputs(arm, frame, emb, train)
    route = table.route_targets().to_numpy(dtype=np.float32)
    served, _thr, _fit = cv._served(arm, x, route, train, frame)
    test = frame[~train][["dataset", "query_id", "query", "shape", "serve",
                          "score_dense_only", "score_sparse_only", "score_pure_rrf"]].copy()
    test["served"] = served
    return test

LANE = FAIR_LANE                    # <- override with a lane from EYE TEST A
dd = query_drilldown(table, LANE)

differ = dd[(dd["shape"] == "routes_differ") & dd["serve"].notna()]
wrong = differ[differ["served"] != differ["serve"]]
print(f"lane={LANE} (cascade): "
      f"{len(dd)} test rows | {len(differ)} decisive | router wrong on {len(wrong)} "
      f"({len(wrong)/max(len(differ),1):.0%})")
print(f"served mix: {dd['served'].value_counts(normalize=True).round(2).to_dict()}  "
      f"vs truth: {differ['serve'].value_counts(normalize=True).round(2).to_dict()}")
pd.set_option("display.max_colwidth", 90)
print("\ndecisive rows where router != truth (read the queries):")
display(wrong[["query", "serve", "served",
               "score_dense_only", "score_sparse_only", "score_pure_rrf"]].head(20))

fit:   0%|          | 0/200 [00:00<?, ?it/s]

lane=rarb-math (cascade): 6676 test rows | 4156 decisive | router wrong on 2435 (59%)
served mix: {'dense_only': 0.89, 'sparse_only': 0.11}  vs truth: {'sparse_only': 0.59, 'dense_only': 0.4, 'pure_rrf': 0.01}

decisive rows where router != truth (read the queries):


,query,serve,served,score_dense_only,score_sparse_only,score_pure_rrf
15372,Problem: A rubber ball is dropped from a height of 100 feet. Each the time ball bounc...,sparse_only,dense_only,0.189279,0.000000,0.116056
15375,Problem: Find all the integer roots of $2x^4 + 4x^3 - 5x^2 + 2x - 3 = 0.$ Enter all t...,sparse_only,dense_only,0.189279,0.000000,0.129203
15445,"Problem: If $P(x)$ is a polynomial in $x,$ and\n\[x^{23} + 23x^{17} - 18x^{16} - 24x^{...",sparse_only,dense_only,0.090309,0.000000,0.000000
15448,Problem: The graph of $y = f(x)$ is shown below.\n\n[asy]\nunitsize(0.5 cm);\n\nreal f...,sparse_only,dense_only,0.189279,0.000000,0.150000
15464,Problem: Find the sum of all the roots of\n\[\frac{x^2 - 13x + 22}{x^2 - 8x + 12} = 0.\],sparse_only,dense_only,0.164272,0.000000,0.000000
15473,"Problem: Let $a,$ $b,$ and $c$ be nonzero real numbers such that $\frac{1}{a} + \frac{...",sparse_only,dense_only,0.150000,0.000000,0.106862
15476,"Problem: Let $a,$ $b,$ and $c$ be the roots of $x^3 + 7x^2 - 11x - 2 = 0.$ Find $a + ...",sparse_only,dense_only,0.189279,0.000000,0.129203
15487,"Problem: The function $f(x) = -3x^2 + 36x - 7,$ defined for all real numbers, does not...",sparse_only,dense_only,0.116056,0.000000,0.090309
15492,"Problem: Let $a,$ $b,$ and $c$ be distinct real numbers. Simplify the expression\n\[\...",sparse_only,dense_only,0.189279,0.000000,0.150000
15493,"Problem: Let $a,$ $b,$ and $c$ be the roots of $2x^3 + 3x^2 + 4x + 5 = 0.$ Find $abc +...",sparse_only,dense_only,0.150000,0.000000,0.129203


## 4c. Load a saved arm & classify (no training here — models come from §3b)

`load_classifier(BASE/"<arm>")` reloads one arm's exact saved weights into `classify(queries)`.
Serve-safe arms (`no_branches` / `shuffled_targets` / `zipf_input`) classify raw strings;
`features_input` / `lightgbm` are saved but need the taxonomy extractor at serve time
(`serve_safe=False`), so raw-query classify is blocked. Serving policy stays gate + constant.

In [22]:
# 4c — load a saved arm; serve by tuned THRESHOLDS, then hedge close calls to pure_rrf
from sentence_transformers import SentenceTransformer

RRF_DELTA = 0.07   # override to pure_rrf when |p_dense - p_sparse| < RRF_DELTA (hedge on top of thresholds)

def serve_thr_rrf(probs, thr, delta=RRF_DELTA):
    """Tuned-threshold serving (unchanged), with a pure_rrf hedge overlaid on near-ties."""
    base = np.asarray(serve_from_probabilities(probs, thr))
    d = probs["dense_only"].to_numpy(); s = probs["sparse_only"].to_numpy()
    return np.where(np.abs(d - s) < delta, "pure_rrf", base)

def load_classifier(arm_dir):
    """Reload ONE arm's saved model into classify(queries) — no retraining."""
    p = _P(arm_dir); meta = _json.loads((p / "meta.json").read_text())
    svd = NgramSvd.load(p / "svd.joblib"); thr = np.load(p / "thresholds.npy")
    st = SentenceTransformer(meta["embedding_model"]); prefix = meta["prefix"]
    if meta["learner"] == "lgbm":
        models = joblib.load(p / "lgbm.joblib")
        prob = lambda X: pd.DataFrame(np.column_stack([m.predict_proba(X)[:, 1] for m in models]), columns=HEAD_ROUTES)
    else:
        er = EncoderRouter.load(p / "router.pt"); prob = er.probabilities
    zstat = (np.load(p / "zipf_mean.npy"), np.load(p / "zipf_std.npy")) if meta["zipf_inputs"] else None

    def classify(queries, delta=RRF_DELTA):
        if not meta["serve_safe"]:
            raise RuntimeError(f"{meta['arm']} uses taxonomy feature_inputs — assemble features first")
        q = [queries] if isinstance(queries, str) else list(queries)
        blocks = [np.asarray(st.encode([prefix + t for t in q], normalize_embeddings=True)),
                  svd.transform(pd.Series(q))]
        if zstat is not None:
            z = ZipfStats().frame(pd.Series(q)).to_numpy(np.float32)
            blocks.append((z - zstat[0]) / zstat[1])
        probs = prob(np.concatenate(blocks, axis=1).astype(np.float32))
        return probs.assign(route=serve_thr_rrf(probs, thr, delta), query=q)
    return classify

classify = load_classifier(BASE / "no_branches")   # reload any arm by name
pd.set_option("display.max_colwidth", 70)
display(classify([
    "how do I refresh laravel migrations programmatically",
    "United States v. Fumo evidence Pennsylvania Ethics Act admissibility",
    "what is the difference between computer engineering and computer science",
]))

,sparse_only,dense_only,route,query
0,0.575483,0.538529,pure_rrf,how do I refresh laravel migrations programmatically
1,0.678473,0.407830,sparse_only,United States v. Fumo evidence Pennsylvania Ethics Act admissibility
2,0.559562,0.789859,sparse_only,what is the difference between computer engineering and computer s...


## 4d. Diagnostic battery — probe the dense/sparse decision boundary

Curated cases grouped by the mechanism they stress. `expect` is a mechanism-based HEURISTIC
(rare identifiers/codes → sparse; conceptual NL → dense; concept+rare-term → hybrid), NOT ground
truth — the interesting rows are the DIVERGENCES. The load-bearing category is `buried-id`: a rare
discriminating token inside fluent text. If the router serves dense there, it is reading fluency
and missing the rare token — the router-rarity-gap made visible on queries you control.

## 2. Build the training table from the cascade labels

`AcceptabilityLabels` derives the existing `ok_*` targets, the cost-aware serving decision, and the outcome shape from the cascade score vector — nothing here depends on the compact cascade artifact carrying a `shape` column.

In [18]:
# 4d — diagnostic battery: does the router use lexical rarity, or default to dense on fluent text?
CASES = [
    ("id/code",   "sparse_only", "CVE-2021-44228 log4j remote code execution"),
    ("id/code",   "sparse_only", "ORA-00942 table or view does not exist"),
    ("id/code",   "sparse_only", "kubectl pod CrashLoopBackOff exit code 137"),
    ("id/code",   "sparse_only", "pip ImportError libGL.so.1 cannot open shared object file"),
    ("id/code",   "sparse_only", "doi:10.1038/s41586-020-2649-2"),
    ("buried-id", "sparse_only", "why does my postgres connection fail with ECONNREFUSED 127.0.0.1:5432"),
    ("buried-id", "sparse_only", "what causes the E11000 duplicate key error in mongodb"),
    ("buried-id", "sparse_only", "how do I fix Objects are not valid as a React child"),
    ("concept",   "dense_only",  "how does photosynthesis differ from cellular respiration"),
    ("concept",   "dense_only",  "what are the ethical implications of autonomous weapons"),
    ("concept",   "dense_only",  "explain the difference between empathy and sympathy"),
    ("concept",   "dense_only",  "why do people procrastinate even when they know the costs"),
    ("mixed",     "pure_rrf",    "difference between BM25 and TF-IDF for document ranking"),
    ("mixed",     "pure_rrf",    "how does the mRNA COVID-19 vaccine trigger an immune response"),
    ("mixed",     "pure_rrf",    "is Rust actually memory safe compared to C++"),
    ("exact",     "sparse_only", "\"I have a dream\" which speech and what year"),
    ("exact",     "sparse_only", "lyrics never gonna give you up never gonna let you down"),
    ("rare-1tok", "sparse_only", "defenestration"),
    ("oov",       "sparse_only", "was ist die Hauptstadt von Osterreich"),
    ("code",      "sparse_only", "for i in range(len(arr)): arr[i] += 1"),
    ("common",    "dense_only",  "what is the best way to be happy in life"),
    ("common",    "dense_only",  "how can I get better at my job over time"),
]
_cats = pd.DataFrame(CASES, columns=["category", "expect", "query"])
_res = classify(_cats["query"].tolist())
_pcols = [c for c in _res.columns if c in ("dense_only", "sparse_only", "pure_rrf")]
_res.insert(0, "category", _cats["category"].to_numpy())
_res.insert(1, "expect", _cats["expect"].to_numpy())
_res["match"] = _res["route"] == _res["expect"]

# CONSTANT baselines vs the SAME heuristic: a sparse-default scores well because the battery is sparse-heavy.
router_match = _res["match"].mean()
const_match = {r: float((_cats["expect"] == r).mean()) for r in ("sparse_only", "dense_only", "pure_rrf")}
best_const_route = max(const_match, key=const_match.get)
print(f"router match vs heuristic: {router_match:.0%}")
print(f"constant baselines: " + ", ".join(f"always-{r} {v:.0%}" for r, v in const_match.items()))
print(f"router LIFT over best constant (always-{best_const_route} {const_match[best_const_route]:.0%}): "
      f"{router_match - const_match[best_const_route]:+.0%}   <- this is the real signal, not the raw match")
print(f"router served mix: {_res['route'].value_counts(normalize=True).round(2).to_dict()} "
      f"(one route dominating => constant-in-disguise)")
pd.set_option("display.max_colwidth", 62)
display(_res[["category", "query", "expect", "route", "match"] + _pcols].round(3))
print("\nbalanced per-category match (buried-id low = rarity blindness):")
display(_res.groupby("category")["match"].agg(["mean", "count"]).round(2))

router match vs heuristic: 64%
constant baselines: always-sparse_only 59%, always-dense_only 27%, always-pure_rrf 14%
router LIFT over best constant (always-sparse_only 59%): +5%   <- this is the real signal, not the raw match
router served mix: {'sparse_only': 0.59, 'pure_rrf': 0.23, 'dense_only': 0.18} (one route dominating => constant-in-disguise)


,category,query,expect,route,match,sparse_only,dense_only
0,id/code,CVE-2021-44228 log4j remote code execution,sparse_only,sparse_only,True,0.457,0.657
1,id/code,ORA-00942 table or view does not exist,sparse_only,sparse_only,True,0.484,0.604
2,id/code,kubectl pod CrashLoopBackOff exit code 137,sparse_only,pure_rrf,False,0.635,0.629
3,id/code,pip ImportError libGL.so.1 cannot open shared object file,sparse_only,sparse_only,True,0.557,0.472
4,id/code,doi:10.1038/s41586-020-2649-2,sparse_only,sparse_only,True,0.545,0.441
5,buried-id,why does my postgres connection fail with ECONNREFUSED 127...,sparse_only,sparse_only,True,0.435,0.528
6,buried-id,what causes the E11000 duplicate key error in mongodb,sparse_only,sparse_only,True,0.569,0.453
7,buried-id,how do I fix Objects are not valid as a React child,sparse_only,sparse_only,True,0.508,0.641
8,concept,how does photosynthesis differ from cellular respiration,dense_only,sparse_only,False,0.437,0.684
9,concept,what are the ethical implications of autonomous weapons,dense_only,sparse_only,False,0.553,0.430



balanced per-category match (buried-id low = rarity blindness):


,mean,count
category,,
buried-id,1.00,3
code,0.00,1
common,0.50,2
concept,0.50,4
exact,1.00,2
id/code,0.80,5
mixed,0.67,3
oov,0.00,1
rare-1tok,0.00,1


In [19]:
display(classify([
    'Who likes Curling?',
    'what are the side effects of DHA',
    'CVE-2021-44228 log4j',
    'http://localhost.com',
    "why do cats purr"
]))

,sparse_only,dense_only,route,query
0,0.991906,0.002932,sparse_only,Who likes Curling?
1,0.378254,0.664700,sparse_only,what are the side effects of DHA
2,0.345968,0.667698,dense_only,CVE-2021-44228 log4j
3,0.416687,0.644957,sparse_only,http://localhost.com
4,0.415303,0.720836,sparse_only,why do cats purr


## 4e. Held-out evaluation — score ALL 6 saved arms (no refitting)

Loads every arm’s saved artifact and scores it on `FAIR_LANE`’s decisive cascade rows. Feature/LightGBM arms are not raw-query serve-safe, but their held-out rows can still assemble the saved feature/Zipf transforms. The check `meta.holdout_lane == FAIR_LANE` prevents leakage; the deployable comparison is the global constant from the training lanes.

In [20]:
# 4e — score every SAVED arm on FAIR_LANE; compare to the per-lane ORACLE and the GLOBAL constant
RRF_DELTA = globals().get("RRF_DELTA", 0.15)
def _serve(probs, thr, delta=RRF_DELTA):        # tuned thresholds + pure_rrf hedge on near-ties
    base = np.asarray(serve_from_probabilities(probs, thr))
    d = probs["dense_only"].to_numpy(); sp = probs["sparse_only"].to_numpy()
    return np.where(np.abs(d - sp) < delta, "pure_rrf", base)

def global_constant(table, holdout_lane):
    """DEPLOYABLE baseline: single best route over ALL training lanes (no lane label at serve time)."""
    tr = table.frame[(table.frame["dataset"] != holdout_lane)
                     & (table.frame["shape"] == "routes_differ") & table.frame["serve"].notna()]
    means = {r: tr[f"score_{r}"].mean() for r in ROUTES}
    return max(means, key=means.get)

def evaluate_saved_frame(arm_dir, table, holdout_lane, global_route):
    p = _P(arm_dir); meta = _json.loads((p / "meta.json").read_text())
    assert meta.get("holdout_lane") == holdout_lane, (
        f"{p.name}: saved holdout={meta.get('holdout_lane')!r} != {holdout_lane!r} — retrain (leakage)")
    svd = NgramSvd.load(p / "svd.joblib"); thr = np.load(p / "thresholds.npy")
    if meta["learner"] == "lgbm":
        models = joblib.load(p / "lgbm.joblib")
        prob = lambda X: pd.DataFrame(np.column_stack([m.predict_proba(X)[:, 1] for m in models]), columns=HEAD_ROUTES)
    else:
        er = EncoderRouter.load(p / "router.pt"); prob = er.probabilities

    frame = table.frame.reset_index(drop=True)
    emb = QueryEmbeddings(meta["embedding_model"]).matrix(frame)
    mask = ((frame["dataset"] == holdout_lane) & (frame["shape"] == "routes_differ")
            & frame["serve"].notna()).to_numpy()
    idx = np.flatnonzero(mask); q = frame.loc[idx, "query"]
    blocks = [emb[idx], svd.transform(q)]
    if meta["feature_inputs"]:
        f = table.feature_matrix.to_numpy(np.float32)[idx]
        blocks.append((f - np.load(p / "feat_mean.npy")) / np.load(p / "feat_std.npy"))
    if meta["zipf_inputs"]:
        z = ZipfStats().frame(q).to_numpy(np.float32)
        blocks.append((z - np.load(p / "zipf_mean.npy")) / np.load(p / "zipf_std.npy"))
    probs = prob(np.concatenate(blocks, axis=1).astype(np.float32))
    served = _serve(probs, thr)

    tf = frame.loc[idx]; sc = {r: tf[f"score_{r}"].to_numpy() for r in ROUTES}
    cap = np.array([sc[r][i] for i, r in enumerate(served)]).mean()
    consts = {r: sc[r].mean() for r in ROUTES}
    lane_best = max(consts, key=consts.get)          # per-lane ORACLE (needs lane label -> NOT deployable)
    lane_c = consts[lane_best]; glob_c = consts[global_route]   # GLOBAL constant (deployable everywhere)
    orc = np.column_stack([sc[r] for r in ROUTES]).max(1).mean()
    hr = lambda base: (cap - base) / (orc - base) if (orc - base) > 0.02 else np.nan
    return {"arm": p.name, "n_dec": len(idx), "captured": cap,
            "lane_route": lane_best, "lane_const": lane_c, "hr_vs_lane": hr(lane_c),
            "global_route": global_route, "global_const": glob_c,
            "lift_vs_global": cap - glob_c, "hr_vs_global": hr(glob_c),
            "oracle": orc, "served_max": float(pd.Series(served).value_counts(normalize=True).max())}

GLOBAL_ROUTE = global_constant(table, FAIR_LANE)
print(f"held-out {FAIR_LANE} | GLOBAL constant (deployable) = {GLOBAL_ROUTE!r} | "
      f"per-lane oracle route may differ")
rows = []
for a in PROBE_ARMS:
    d = BASE / a.name
    if not (d / "meta.json").exists():
        print(f"[{a.name}] not saved yet — run §3b"); continue
    try:
        rows.append(evaluate_saved_frame(d, table, FAIR_LANE, GLOBAL_ROUTE))
    except AssertionError as e:
        print("SKIP:", e)
print("\nhr_vs_global = the DEPLOYABLE bar (router beats the constant you could actually ship);")
print("hr_vs_lane   = the ORACLE bar (per-lane constant needs a lane label -> not shippable):")
display(pd.DataFrame(rows).round(3) if rows else "no saved models yet — run §3b")

held-out rarb-math | GLOBAL constant (deployable) = 'pure_rrf' | per-lane oracle route may differ

hr_vs_global = the DEPLOYABLE bar (router beats the constant you could actually ship);
hr_vs_lane   = the ORACLE bar (per-lane constant needs a lane label -> not shippable):


,arm,n_dec,captured,lane_route,lane_const,hr_vs_lane,global_route,global_const,lift_vs_global,hr_vs_global,oracle,served_max
0,no_branches,4156,0.426,pure_rrf,0.492,-0.281,pure_rrf,0.492,-0.066,-0.281,0.725,0.629
1,shuffled_targets,4156,0.492,pure_rrf,0.492,0.000,pure_rrf,0.492,0.000,0.000,0.725,1.000
2,features_input_nocorpus,4156,0.440,pure_rrf,0.492,-0.220,pure_rrf,0.492,-0.051,-0.220,0.725,0.717
3,zipf_input_nocorpus,4156,0.433,pure_rrf,0.492,-0.251,pure_rrf,0.492,-0.059,-0.251,0.725,0.748
4,lightgbm_nocorpus,4156,0.455,pure_rrf,0.492,-0.156,pure_rrf,0.492,-0.037,-0.156,0.725,0.970
5,design_branches,4156,0.417,pure_rrf,0.492,-0.321,pure_rrf,0.492,-0.075,-0.321,0.725,0.495


## 4f. Zipf-rule router — the simplest thing (serve-safe, no model)

The learned arms can be compared with the constant (§4e), so state the mapping instead of learning it: a
3-threshold rule over `ZipfStats` (background word rarity). Rare/OOV tokens (CVE, log4j, error codes)
→ sparse; all-common words → dense; the ambiguous middle → the base constant. Deterministic,
interpretable, serve-safe (wordfreq bundled). Held-out headroom over the best constant is the same
bar as every learned arm — the curated demo will look great, so trust the §4f held-out number, not it.

In [ ]:
# 4f — pure Zipf-rule router: rare -> sparse, common -> dense, else base. No model. Serve-safe.
from encoder_router.table import ZipfStats

Z_RARE_SHARE  = 0.30    # >= this share of rare (zipf<3) tokens -> sparse (lexical / identifiers)
Z_OOV_SHARE   = 0.20    # >= this share of OOV tokens          -> sparse
Z_COMMON_MEAN = 4.50    # mean zipf >= this (all-common words) -> dense (semantic)
Z_BASE        = "sparse_only"   # ambiguous middle -> the per-lane constant

def load_classifier_with_zipf(rare=Z_RARE_SHARE, oov=Z_OOV_SHARE, common=Z_COMMON_MEAN, base=Z_BASE):
    zs = ZipfStats()
    def classify(queries):
        q = [queries] if isinstance(queries, str) else list(queries)
        zf = zs.frame(pd.Series(q))
        rs = zf["zipf.rare_share"].to_numpy(); ov = zf["zipf.oov_share"].to_numpy(); mn = zf["zipf.mean"].to_numpy()
        route = np.full(len(q), base, dtype=object)
        route = np.where(mn >= common, "dense_only", route)                 # all common -> dense
        route = np.where((rs >= rare) | (ov >= oov), "sparse_only", route)  # rare/oov -> sparse (wins)
        return pd.DataFrame({"query": q, "route": route}).join(zf.round(2))
    return classify

def evaluate_zipf_rule(table, holdout_lane, **kw):
    clf = load_classifier_with_zipf(**kw)
    tf = table.frame[(table.frame["dataset"] == holdout_lane)
                     & (table.frame["shape"] == "routes_differ") & table.frame["serve"].notna()]
    served = clf(tf["query"].tolist())["route"].to_numpy()
    sc = {r: tf[f"score_{r}"].to_numpy() for r in ROUTES}
    cap = np.array([sc[r][i] for i, r in enumerate(served)]).mean()
    consts = {r: sc[r].mean() for r in ROUTES}
    best = max(consts, key=consts.get); orc = np.column_stack([sc[r] for r in ROUTES]).max(1).mean()
    return {"rule": "zipf", "n_decisive": len(tf), "best_route": best,
            "headroom": (cap - consts[best]) / (orc - consts[best]) if (orc - consts[best]) > 0.02 else np.nan,
            "raw_lift": cap - consts[best], "captured": cap, "best_const": consts[best], "oracle": orc,
            "served_max": float(pd.Series(served).value_counts(normalize=True).max())}

zipf_router = load_classifier_with_zipf()
pd.set_option("display.max_colwidth", 55)
print("demo (looks great by construction — not the verdict):")
display(zipf_router([
    "CVE-2021-44228 log4j remote code execution", "http://localhost.com", "Who likes Curling?",
    "what are the side effects of DHA", "how does photosynthesis differ from cellular respiration",
]))
print(f"\nheld-out on {FAIR_LANE} (the real bar — beat best_const, served_max<0.95):")
display(pd.DataFrame([evaluate_zipf_rule(table, FAIR_LANE)]).round(3))